Import ชุดคำสั่งที่จำเป็น

In [1]:
import pandas as pd
import numpy as np

# for reading and displaying images
from skimage.io import imread
import matplotlib.pyplot as plt

# for creating validation set
from sklearn.model_selection import train_test_split

# for evaluating the model
from sklearn.metrics import accuracy_score
from tqdm import tqdm

# PyTorch libraries and modules
import torch
from torch.nn import CrossEntropyLoss
from torch.optim import Adam, SGD


Data Loader

In [2]:
from google.colab import drive
drive.mount('/content/drive')

!unzip -q "/content/drive/MyDrive/Deep Learning/ThaiCharacter Dataset.zip" -d /content/dataset

# จากนั้นตั้งตัวแปร path สำหรับใช้ต่อในโค้ด:

DATA_PATH = "/content/dataset/round2"

Mounted at /content/drive


In [3]:
import os

DATA_PATH = "/content/dataset/round2"
classes = sorted([d for d in os.listdir(DATA_PATH) if os.path.isdir(os.path.join(DATA_PATH, d))])

Data Loader + Data Augmentation (วนอ่านภาพจริงพร้อมเติมภาพให้คลาสที่ขาด)

In [4]:
from skimage.transform import resize, rotate
from skimage.util import random_noise

def augment_image(img):
    angle = np.random.uniform(-15, 15)  # เทียบเท่า RandomRotation
    img_aug = rotate(img, angle, mode='edge')
    if np.random.rand() < 0.5:
        img_aug = random_noise(img_aug, var=0.005)  # เพิ่ม noise เบาๆ
    return img_aug

In [5]:
MIN_SAMPLES_PER_CLASS = 50

train_img = []
train_label = []

for cls in tqdm(classes):
    cls_folder = os.path.join(DATA_PATH, cls)
    img_files = [f for f in os.listdir(cls_folder) if f.lower().endswith('.jpg')]

    original_imgs = []
    for fname in img_files:
        img = imread(os.path.join(cls_folder, fname), as_gray=True)
        img = resize(img, (32, 32), preserve_range=True)  # preserve_range=True ป้องกันการหารซ้ำ
        img /= 255.0
        original_imgs.append(img.astype('float32'))

    train_img.extend(original_imgs)
    train_label.extend([cls] * len(original_imgs))

    # เติมภาพด้วย Augmentation เฉพาะคลาสที่ขาด
    n_needed = MIN_SAMPLES_PER_CLASS - len(original_imgs)
    if n_needed > 0:
        for i in range(n_needed):
            base_img = original_imgs[i % len(original_imgs)]
            train_img.append(augment_image(base_img).astype('float32'))
            train_label.append(cls)

train_x = np.array(train_img)
train_y = np.array(train_label)
print(train_x.shape)

100%|██████████| 72/72 [00:53<00:00,  1.35it/s]


(63317, 32, 32)


Training & Validating Set Generation

In [6]:
# แปลง label เป็น class index ก่อน split (ต้องทำก่อน ไม่งั้น val_y จะไม่ได้แปลงด้วย)
class_to_idx = {cls: idx for idx, cls in enumerate(classes)}  # classes จากตอน Data Loader
train_y = np.array([class_to_idx[l] for l in train_y])

train_x, val_x, train_y, val_y = train_test_split(train_x, train_y, test_size=0.2)  # แบ่ง 80/20

# converting training images into torch format
train_x = train_x.reshape(-1, 1, 32, 32)  # เราไม่รู้จำนวนที่แน่นอนล่วงหน้าต้องใช้ -1 แทน
train_x = torch.from_numpy(train_x).to(torch.float32)
train_y = torch.from_numpy(train_y).to(torch.long)  # CrossEntropyLoss ต้องการ target เป็น torch.long

# shape of training data
print(train_x.shape, train_y.shape)

# converting validation images into torch format
val_x = val_x.reshape(-1, 1, 32, 32)
val_x = torch.from_numpy(val_x).to(torch.float32)
val_y = torch.from_numpy(val_y).to(torch.long)

# shape of validation data
print(val_x.shape, val_y.shape)

torch.Size([50653, 1, 32, 32]) torch.Size([50653])
torch.Size([12664, 1, 32, 32]) torch.Size([12664])


Model Loader

In [7]:
%%writefile Net.py
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

class Net(nn.Module):
    def __init__(self, num_classes=72):
        super().__init__()
        self.backbone = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        self.backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.backbone.fc = nn.Linear(self.backbone.fc.in_features, num_classes)

    def forward(self, x):
        return self.backbone(x)

Writing Net.py


In [8]:
from Net import Net
model = Net()
print(model)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 162MB/s]


Net(
  (backbone): ResNet(
    (conv1): Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_runnin

Defining Learning Algorithm

In [9]:
# defining the model
model = Net(num_classes=len(classes))

# defining the optimizer
optimizer = Adam(model.parameters(), lr=0.0001)

# defining the loss function
criterion = CrossEntropyLoss()

# force using 'cuda'
device = torch.device('cuda')
model = model.to(device)
criterion = criterion.to(device)

print(model)

Net(
  (backbone): ResNet(
    (conv1): Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_runnin

Training Model

In [10]:
# empty list to store training losses/accuracy
train_losses = []
train_accuracies = []

# empty list to store validation losses/accuracy
val_losses = []
val_accuracies = []

# defining the number of epochs
n_epochs = 25

from torch.utils.data import TensorDataset, DataLoader

train_loader = DataLoader(TensorDataset(train_x, train_y), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(val_x, val_y), batch_size=64)

for epoch in tqdm(range(n_epochs)):
    model.train()
    tr_loss = 0
    tr_correct = 0
    tr_total = 0

    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)

        # clearing the Gradients of the model parameters
        optimizer.zero_grad()

        # prediction for training set
        output_train = model(x_batch)

        loss_train = criterion(output_train, y_batch)
        loss_train.backward()
        optimizer.step()
        tr_loss += loss_train.item()

        # accuracy ของ training batch นี้
        predicted = torch.argmax(output_train, dim=1)
        tr_correct += (predicted == y_batch).sum().item()
        tr_total += y_batch.size(0)

    train_losses.append(tr_loss / len(train_loader))
    train_accuracies.append(tr_correct / tr_total)

    # evaluating performance on the validation set
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            output_val = model(x_batch)
            val_loss += criterion(output_val, y_batch).item()

            # accuracy ของ validation batch นี้
            predicted = torch.argmax(output_val, dim=1)
            val_correct += (predicted == y_batch).sum().item()
            val_total += y_batch.size(0)

    val_losses.append(val_loss / len(val_loader))
    val_accuracies.append(val_correct / val_total)

    print(f"Epoch {epoch+1}/{n_epochs} - train_loss: {train_losses[-1]:.4f} - train_acc: {train_accuracies[-1]*100:.2f}% - val_loss: {val_losses[-1]:.4f} - val_acc: {val_accuracies[-1]*100:.2f}%")

  4%|▍         | 1/25 [00:22<09:00, 22.52s/it]

Epoch 1/25 - train_loss: 0.3525 - train_acc: 91.57% - val_loss: 0.1052 - val_acc: 97.07%


  8%|▊         | 2/25 [00:44<08:25, 22.00s/it]

Epoch 2/25 - train_loss: 0.0932 - train_acc: 97.30% - val_loss: 0.0841 - val_acc: 97.62%


 12%|█▏        | 3/25 [01:06<08:07, 22.14s/it]

...

Epoch 23/25 - train_loss: 0.0191 - train_acc: 99.35% - val_loss: 0.0855 - val_acc: 98.20%


 96%|█████████▌| 24/25 [09:53<00:25, 25.22s/it]

Epoch 24/25 - train_loss: 0.0181 - train_acc: 99.35% - val_loss: 0.0855 - val_acc: 98.44%


100%|██████████| 25/25 [10:18<00:00, 24.75s/it]

Epoch 25/25 - train_loss: 0.0196 - train_acc: 99.31% - val_loss: 0.0860 - val_acc: 98.48%


Save Model

In [12]:
# A common PyTorch convention is to save models using either a .pt or .pth file extension.
torch.save(model.state_dict(), '/content/drive/MyDrive/Deep Learning/model.pt')